# dataset

> A generic `torch` dataset over the patch layout, hard-example sampling weights, and dataloader construction.

The dataset reads the per-channel patch files, assembles the model input with `channels.build_image`, and returns an `(image, mask)` pair. Binary and multi-class differ only in how the mask is decoded. This module also fixes the original script's missing `build_datasets` helper that `build_dataloaders` relied on.

In [1]:
#| default_exp dataset

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from segmentation_tools_pt.channels import build_image, DEFAULT_CHANNELS
from segmentation_tools_pt.masks import binarize
from segmentation_tools_pt.transforms import get_train_transforms, get_val_transforms

## The dataset

One sample on disk is a set of channel files plus a mask. `__getitem__` loads them, builds the multi-channel image, decodes the mask, and applies the albumentations transform that keeps image and mask aligned.

Mask decoding depends on `mask_mode`:
- `single` (default) — one mask file; binary -> `{0,1}`, multi-class -> integer labels. Unchanged from before.
- `stacked` — one `K`-channel PNG per sample (`ch0=chip, ch1=void, ch2=lifted`, ...); each channel is independently binarized, so overlap (e.g. void inside chip) is preserved.
- `per_class_files` — `K` separate single-channel PNGs per sample (`{sid}_{class_name}.png`), one per entry in `class_names`.

`stacked`/`per_class_files` both return an `(H, W, K)` float mask. Albumentations' default `mask` target applies the same spatial transform to any trailing channel count, so the existing crop/flip/affine pipeline needs no changes -- but `ToTensorV2` does **not** transpose masks the way it does images (verified empirically: a `(H,W,K)` mask stays `(H,W,K)` after `ToTensorV2`, unlike the image which becomes `(C,H,W)`), so `__getitem__` permutes it to `(K,H,W)` itself.

In [4]:
#| export
def _decode_stacked_mask(arr: np.ndarray, mask_channels: int) -> np.ndarray:
    "arr: (H,W) or (H,W,>=mask_channels) uint8 from cv2.IMREAD_UNCHANGED. Returns (H,W,mask_channels) float32 {0,1}."
    if arr.ndim == 2:
        arr = arr[:, :, None]
    assert arr.shape[2] >= mask_channels, f'expected >= {mask_channels} mask channels, got {arr.shape[2]}'
    return (arr[:, :, :mask_channels] > 127).astype(np.float32)


def _load_per_class_masks(mask_dir: Path, sid: str, class_names: Sequence[str], ext: str = '.png') -> np.ndarray:
    "Read one file per class ({sid}_{class_name}{ext}); returns (H,W,K) float32 {0,1}."
    chans = []
    for name in class_names:
        p = mask_dir / f'{sid}_{name}{ext}'
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        assert m is not None, f'Missing mask channel {name!r} for {sid} at {p}'
        chans.append((m > 127).astype(np.float32))
    return np.stack(chans, axis=-1)

In [5]:
#| export
class SegmentationDataset(Dataset):
    """Loads patch samples produced by `dataset_build`.

    Returns `{'image': (C,H,W) float, 'mask': (H,W) long or (K,H,W) float, 'sample_id': str}` --
    the mask is `(H,W) long` for `mask_mode='single'` (binary/multi-class) and
    `(K,H,W) float {0,1}` for `mask_mode='stacked'`/`'per_class_files'` (multilabel).
    """

    def __init__(
        self,
        root: str,                                   # dataset root (has train/ val/ metadata.json)
        split: str = 'train',                        # 'train' or 'val'
        num_classes: int = 1,                        # 1 = binary, >1 = multi-class (only used when mask_mode='single')
        n_source_channels: int = 2,                  # number of raw channel pngs per sample
        channel_names: Sequence[str] = DEFAULT_CHANNELS,  # how to assemble the model image
        transform=None,                              # albumentations Compose or None
        mask_mode: str = 'single',                   # 'single' | 'stacked' | 'per_class_files'
        mask_channels: Optional[int] = None,          # K, required for 'stacked'
        class_names: Optional[Sequence[str]] = None,  # K class name suffixes, required for 'per_class_files'
    ):
        assert mask_mode in ('single', 'stacked', 'per_class_files'), f'unknown mask_mode {mask_mode!r}'
        if mask_mode == 'stacked':
            assert mask_channels is not None, "mask_mode='stacked' requires mask_channels"
        if mask_mode == 'per_class_files':
            assert class_names is not None, "mask_mode='per_class_files' requires class_names"
        self.root = Path(root)
        self.split = split
        self.num_classes = num_classes
        self.n_source_channels = n_source_channels
        self.channel_names = list(channel_names)
        self.transform = transform
        self.mask_mode = mask_mode
        self.mask_channels = mask_channels
        self.class_names = list(class_names) if class_names is not None else None
        self.img_dir = self.root / split / 'images'
        self.mask_dir = self.root / split / 'masks'
        meta_path = self.root / 'metadata.json'
        if meta_path.exists():
            self.metadata = json.loads(meta_path.read_text())[split]
        else:
            ids = (self.root / f'{split}.txt').read_text().split()
            self.metadata = [{'sample_id': s, 'fg_pixels': -1, 'is_empty': False} for s in ids]
        self.sample_ids = [m['sample_id'] for m in self.metadata]

    def __len__(self) -> int:
        return len(self.sample_ids)

    def _load_image(self, sid: str) -> np.ndarray:
        "Read raw channels and assemble the (H, W, C) model input."
        chans = []
        for i in range(self.n_source_channels):
            # Single-channel patches may use the plain patch id; multi-channel use _ch{i}.
            paths = [self.img_dir / f'{sid}_ch{i}.png']
            if i == 0 and self.n_source_channels == 1:
                paths.insert(0, self.img_dir / f'{sid}.png')
            arr = None
            for p in paths:
                arr = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if arr is not None:
                    break
            assert arr is not None, f'Missing channel {i} for {sid}'
            chans.append(arr)
        return build_image(chans, self.channel_names)

    def _load_mask(self, sid: str) -> np.ndarray:
        if self.mask_mode == 'per_class_files':
            return _load_per_class_masks(self.mask_dir, sid, self.class_names)
        # 'single' and 'stacked' both look for one file, same base-name convention.
        for p in [self.mask_dir / f'{sid}.png', self.mask_dir / f'{sid}_mask.png']:
            if self.mask_mode == 'stacked':
                arr = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
                if arr is not None:
                    return _decode_stacked_mask(arr, self.mask_channels)
            else:
                m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if m is not None:
                    return binarize(m) if self.num_classes == 1 else m.astype(np.int64)
        raise FileNotFoundError(f'Missing mask for {sid}')

    def __getitem__(self, idx: int) -> Dict:
        sid = self.sample_ids[idx]
        image, mask = self._load_image(sid), self._load_mask(sid)
        multilabel = self.mask_mode != 'single'
        if self.transform is not None:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            # ToTensorV2 does not transpose masks like it does images; (H,W,K) -> (K,H,W) ourselves.
            mask = aug['mask'].permute(2, 0, 1).float() if multilabel else aug['mask'].long()
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32) / 255.0)
            mask = (torch.from_numpy(mask.transpose(2, 0, 1).astype(np.float32)) if multilabel
                    else torch.from_numpy(mask.astype(np.int64)))
        return {'image': image, 'mask': mask, 'sample_id': sid}

    def get_sample_weights(
        self,
        eps: float = 500.0,      # floor so dense patches are not zero-weighted
        min_fg: int = 500,       # 'boundary' foreground count for empty patches
    ) -> List[float]:
        """Weight hard (sparse-foreground) patches higher for `WeightedRandomSampler`.

        Sparse-foreground patches (partial targets at edges) are harder, so they
        get `1/(fg+eps)`. Empty patches get a neutral boundary weight; unknown
        counts get 1.0.
        """
        boundary = 1.0 / (min_fg + eps)
        out = []
        for m in self.metadata:
            fg = m.get('fg_pixels', -1)
            if fg < 0:
                out.append(1.0)
            elif m.get('is_empty', False):
                out.append(boundary)
            else:
                out.append(1.0 / (fg + eps))
        return out

In [6]:
# test: SegmentationDataset mask_mode='stacked' and 'per_class_files', overlap preserved
import tempfile

rng = np.random.default_rng(0)

def _write_patch_cache(d, n=3, size=20):
    "Minimal hand-written patch cache (bypasses dataset_build/patches -- Phase 1 only tests loading)."
    d = Path(d)
    for split in ('train', 'val'):
        (d / split / 'images').mkdir(parents=True)
        (d / split / 'masks').mkdir(parents=True)
    meta = {'train': [], 'val': []}
    for i in range(n):
        sid = f's{i}'
        cv2.imwrite(str(d / 'train' / 'images' / f'{sid}.png'), rng.integers(0, 256, (size, size), np.uint8))
        meta['train'].append({'sample_id': sid, 'fg_pixels': 10, 'is_empty': False})
    (d / 'metadata.json').write_text(json.dumps(meta))
    return d

with tempfile.TemporaryDirectory() as d:
    d = _write_patch_cache(d)
    # stacked: one 3-channel PNG, void strictly inside chip
    stacked = np.zeros((20, 20, 3), np.uint8)
    stacked[4:16, 4:16, 0] = 255   # chip
    stacked[8:12, 8:12, 1] = 255   # void, inside chip
    stacked[5:15, 5:15, 2] = 255   # lifted
    for i in range(3):
        cv2.imwrite(str(d / 'train' / 'masks' / f's{i}.png'), stacked)
    ds = SegmentationDataset(str(d), 'train', n_source_channels=1, channel_names=['ch0'],
                              mask_mode='stacked', mask_channels=3)
    sample = ds[0]
    assert sample['mask'].shape == (3, 20, 20)
    assert sample['mask'].dtype == torch.float32
    assert bool(sample['mask'][1, 9, 9] == 1) and bool(sample['mask'][0, 9, 9] == 1)  # void@interior, chip also 1
    assert bool(sample['mask'][1, 5, 5] == 0) and bool(sample['mask'][0, 5, 5] == 1)  # chip-only: void 0, chip 1

with tempfile.TemporaryDirectory() as d:
    d = _write_patch_cache(d)
    # per_class_files: one file per class, same overlap layout
    chip = np.zeros((20, 20), np.uint8); chip[4:16, 4:16] = 255
    void = np.zeros((20, 20), np.uint8); void[8:12, 8:12] = 255
    lifted = np.zeros((20, 20), np.uint8); lifted[5:15, 5:15] = 255
    for i in range(3):
        cv2.imwrite(str(d / 'train' / 'masks' / f's{i}_chip.png'), chip)
        cv2.imwrite(str(d / 'train' / 'masks' / f's{i}_void.png'), void)
        cv2.imwrite(str(d / 'train' / 'masks' / f's{i}_lifted.png'), lifted)
    ds = SegmentationDataset(str(d), 'train', n_source_channels=1, channel_names=['ch0'],
                              mask_mode='per_class_files', class_names=['chip', 'void', 'lifted'])
    sample = ds[0]
    assert sample['mask'].shape == (3, 20, 20)
    assert bool(sample['mask'][1, 9, 9] == 1) and bool(sample['mask'][1, 5, 5] == 0)

print('SegmentationDataset ✓  stacked + per_class_files mask_mode, overlap preserved, (K,H,W) float')

SegmentationDataset ✓  stacked + per_class_files mask_mode, overlap preserved, (K,H,W) float


## Build datasets and loaders

`build_datasets` pairs the train/val splits with their transforms (this is the helper the original script called but never defined). `build_dataloaders` wraps them, oversampling hard patches in training via a `WeightedRandomSampler`.

### DataLoader prefetch knob (issue #20)

`num_workers` alone controls parallel loading, but each worker only starts fetching the
next batch once the consumer takes the current one — on network shares (NFS patch caches)
that stalls the GPU between batches. `prefetch_factor` keeps `prefetch_factor` batches in
flight per worker. PyTorch **rejects** `prefetch_factor` when `num_workers == 0`, so both
builder families thread it through `_prefetch_kwargs`: inert (torch's default of 2 per
worker) unless workers exist. Same knob for training loaders (`train.prefetch_factor`) and
production inference loaders (`inference.prefetch_factor`).

Experiment first — pin the semantics this torch build enforces:

In [7]:
# experiment: DataLoader prefetch_factor semantics (why the guarded passthrough exists)
from torch.utils.data import DataLoader, Dataset

class _Nums(Dataset):
    def __len__(self): return 8
    def __getitem__(self, i): return i

dl = DataLoader(_Nums(), batch_size=2, num_workers=0, prefetch_factor=None)
print('num_workers=0, prefetch_factor=None -> OK (knob inert)')
try:
    DataLoader(_Nums(), batch_size=2, num_workers=0, prefetch_factor=2)
    raise AssertionError('torch accepted prefetch_factor with num_workers=0')
except ValueError:
    print('num_workers=0, prefetch_factor=2 -> ValueError (hence the guard)')
dl = DataLoader(_Nums(), batch_size=2, num_workers=2, prefetch_factor=1)
print('num_workers=2, prefetch_factor=1 -> OK; first batch:', next(iter(dl)).tolist())

num_workers=0, prefetch_factor=None -> OK (knob inert)
num_workers=0, prefetch_factor=2 -> ValueError (hence the guard)


num_workers=2, prefetch_factor=1 -> OK; first batch: [0, 1]


In [8]:
#| export
def _seed_worker(worker_id: int) -> None:
    "Seed NumPy/Python RNGs inside each DataLoader worker from torch's worker seed."
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def _make_loader_generator(seed: Optional[int]):
    "Return a torch Generator seeded for reproducible sampling, or None."
    if seed is None:
        return None
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return generator


def _prefetch_kwargs(num_workers: int, prefetch_factor: Optional[int] = None) -> dict:
    "DataLoader kwargs enabling prefetch_factor only when workers exist (torch errors otherwise)."
    return {'prefetch_factor': prefetch_factor} if num_workers > 0 and prefetch_factor is not None else {}


def build_datasets(
    root: str,                                   # dataset root
    num_classes: int = 1,                        # 1 = binary, >1 = multi-class (mask_mode='single' only)
    n_source_channels: int = 2,                  # raw channels per sample
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # model-image assembly
    crop_size: int = 256,                        # crop size for transforms
    mean: Optional[Sequence[float]] = None,      # per-channel mean (defaults to zeros)
    std: Optional[Sequence[float]] = None,       # per-channel std (defaults to ones)
    mask_mode: str = 'single',                   # 'single' | 'stacked' | 'per_class_files'
    mask_channels: Optional[int] = None,          # K, required for 'stacked'
    class_names: Optional[Sequence[str]] = None,  # K class name suffixes, required for 'per_class_files'
) -> Tuple[SegmentationDataset, SegmentationDataset]:
    """Return `(train_ds, val_ds)` with train/val transforms attached."""
    c = len(channel_names)
    mean = tuple(mean) if mean is not None else (0.,) * c
    std = tuple(std) if std is not None else (1.,) * c
    common = dict(num_classes=num_classes, n_source_channels=n_source_channels, channel_names=channel_names,
                  mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names)
    train_ds = SegmentationDataset(root, 'train', transform=get_train_transforms(crop_size, mean, std), **common)
    val_ds = SegmentationDataset(root, 'val', transform=get_val_transforms(crop_size, mean, std), **common)
    return train_ds, val_ds

def build_dataloaders(
    root: str,                  # dataset root
    batch_size: int = 16,       # samples per batch
    num_workers: int = 4,       # dataloader workers (use 2 on network shares)
    prefetch_factor: Optional[int] = None,  # batches kept in flight per worker (needs num_workers > 0); None = torch default
    crop_size: int = 256,       # crop size
    num_classes: int = 1,       # 1 = binary, >1 = multi-class (mask_mode='single' only)
    n_source_channels: int = 2, # raw channels per sample
    channel_names: Sequence[str] = DEFAULT_CHANNELS,  # model-image assembly
    mean: Optional[Sequence[float]] = None,  # per-channel mean
    std: Optional[Sequence[float]] = None,   # per-channel std
    pin_memory: bool = True,    # pin host memory for faster H2D copies
    seed: Optional[int] = None, # run seed for reproducible sampling and worker RNGs
    mask_mode: str = 'single',                   # 'single' | 'stacked' | 'per_class_files'
    mask_channels: Optional[int] = None,          # K, required for 'stacked'
    class_names: Optional[Sequence[str]] = None,  # K class name suffixes, required for 'per_class_files'
) -> Tuple[DataLoader, DataLoader, SegmentationDataset, SegmentationDataset]:
    """Return `(train_loader, val_loader, train_ds, val_ds)`.

    Datasets are returned too so you can inspect samples or re-weight without
    re-instantiating.
    """
    train_ds, val_ds = build_datasets(root, num_classes, n_source_channels, channel_names, crop_size, mean, std,
                                       mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names)
    sampler = WeightedRandomSampler(
        weights=train_ds.get_sample_weights(),
        num_samples=len(train_ds),
        replacement=True,
        generator=_make_loader_generator(seed),
    )
    worker_init_fn = _seed_worker if num_workers > 0 else None
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        **_prefetch_kwargs(num_workers, prefetch_factor),
        pin_memory=pin_memory,
        drop_last=True,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(seed),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        **_prefetch_kwargs(num_workers, prefetch_factor),
        pin_memory=pin_memory,
        drop_last=False,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(None if seed is None else seed + 1),
    )
    return train_loader, val_loader, train_ds, val_ds

## Direct mode (full images, no patch cache)

`DirectDataset` reads from `raw_dir/images/` + `raw_dir/masks/` and applies on-the-fly random/center crop. Use when images are small enough to skip pre-built patches.

In [9]:
#| export
import albumentations as A

from segmentation_tools_pt.dataset_build import resolve_channel_paths, resolve_mask_path, split_sources
from segmentation_tools_pt.masks import fit_rotated_rects_mask


def split_sample_ids(sample_ids, val_fraction, seed):
    "Shuffle and split sample ids into (train_ids, val_ids)."
    return split_sources(sample_ids, val_fraction, seed)


def _wrap_with_pad(transform, crop_size):
    return A.Compose([
        A.PadIfNeeded(min_height=crop_size, min_width=crop_size, border_mode=cv2.BORDER_REFLECT_101),
        *transform.transforms,
    ])


class DirectDataset(Dataset):
    """Load full-resolution images+masks from raw_dir (no patch cache).

    `mask_mode` mirrors `SegmentationDataset`: `'single'` (binary/multi-class,
    default), `'stacked'` (one K-channel PNG), or `'per_class_files'`
    (`{sid}_{class_name}{exts}` per class, under `raw_dir/masks/`).
    """

    def __init__(
        self, raw_dir, sample_ids, num_classes=1, n_source_channels=1,
        channel_names=DEFAULT_CHANNELS, mask_suffix='_mask', exts='.png',
        fit_rects=False, transform=None,
        mask_mode: str = 'single', mask_channels: Optional[int] = None,
        class_names: Optional[Sequence[str]] = None,
    ):
        assert mask_mode in ('single', 'stacked', 'per_class_files'), f'unknown mask_mode {mask_mode!r}'
        if mask_mode == 'stacked':
            assert mask_channels is not None, "mask_mode='stacked' requires mask_channels"
        if mask_mode == 'per_class_files':
            assert class_names is not None, "mask_mode='per_class_files' requires class_names"
        self.raw_dir = Path(raw_dir)
        self.sample_ids = list(sample_ids)
        self.num_classes = num_classes
        self.n_source_channels = n_source_channels
        self.channel_names = list(channel_names)
        self.mask_suffix = mask_suffix
        self.exts = exts
        self.fit_rects = fit_rects
        self.transform = transform
        self.mask_mode = mask_mode
        self.mask_channels = mask_channels
        self.class_names = list(class_names) if class_names is not None else None

    def __len__(self): return len(self.sample_ids)

    def _load_image(self, sid):
        paths = resolve_channel_paths(self.raw_dir, sid, self.n_source_channels, self.exts)
        chans = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in paths]
        for i, c in enumerate(chans):
            assert c is not None, f'Missing channel {i} for {sid}'
        return build_image(chans, self.channel_names)

    def _load_mask(self, sid):
        if self.mask_mode == 'per_class_files':
            return _load_per_class_masks(self.raw_dir / 'masks', sid, self.class_names, self.exts)
        path = resolve_mask_path(self.raw_dir, sid, self.exts, self.mask_suffix)
        if self.mask_mode == 'stacked':
            arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            assert arr is not None, f'Missing mask for {sid}'
            return _decode_stacked_mask(arr, self.mask_channels)
        m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        assert m is not None, f'Missing mask for {sid}'
        if self.fit_rects and self.num_classes == 1:
            m = fit_rotated_rects_mask(m)
        return binarize(m) if self.num_classes == 1 else m.astype(np.int64)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        image, mask = self._load_image(sid), self._load_mask(sid)
        multilabel = self.mask_mode != 'single'
        if self.transform is not None:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            mask = aug['mask'].permute(2, 0, 1).float() if multilabel else aug['mask'].long()
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32) / 255.0)
            mask = (torch.from_numpy(mask.transpose(2, 0, 1).astype(np.float32)) if multilabel
                    else torch.from_numpy(mask.astype(np.int64)))
        return {'image': image, 'mask': mask, 'sample_id': sid}


def build_direct_datasets(
    raw_dir, train_ids, val_ids, num_classes=1, n_source_channels=1,
    channel_names=DEFAULT_CHANNELS, mask_suffix='_mask', exts='.png',
    fit_rects=False, crop_size=256, mean=None, std=None, aug=None, flips=False,
    boundary_blur=None, mask_mode='single', mask_channels=None, class_names=None,
    ):
    c = len(channel_names)
    mean = tuple(mean) if mean is not None else (0.,) * c
    std = tuple(std) if std is not None else (1.,) * c
    common = dict(num_classes=num_classes, n_source_channels=n_source_channels,
                  channel_names=channel_names, mask_suffix=mask_suffix, exts=exts, fit_rects=fit_rects,
                  mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names)
    train_tf = _wrap_with_pad(
        get_train_transforms(crop_size, mean, std, aug=aug, flips=flips, boundary_blur=boundary_blur),
        crop_size,
    )
    val_tf = _wrap_with_pad(get_val_transforms(crop_size, mean, std), crop_size)
    return (
        DirectDataset(raw_dir, train_ids, transform=train_tf, **common),
        DirectDataset(raw_dir, val_ids, transform=val_tf, **common),
    )


def build_direct_dataloaders(
    raw_dir, train_ids, val_ids, batch_size=16, num_workers=4,
    prefetch_factor: Optional[int] = None,  # batches kept in flight per worker (needs num_workers > 0); None = torch default
    crop_size=256,
    num_classes=1, n_source_channels=1, channel_names=DEFAULT_CHANNELS,
    mask_suffix='_mask', exts='.png', fit_rects=False, mean=None, std=None,
    aug=None, flips=False, boundary_blur=None, pin_memory=True, seed: Optional[int] = None,
    mask_mode='single', mask_channels=None, class_names=None,
    ):
    train_ds, val_ds = build_direct_datasets(
        raw_dir, train_ids, val_ids, num_classes, n_source_channels, channel_names,
        mask_suffix, exts, fit_rects, crop_size, mean, std, aug=aug, flips=flips,
        boundary_blur=boundary_blur, mask_mode=mask_mode, mask_channels=mask_channels, class_names=class_names,
    )
    worker_init_fn = _seed_worker if num_workers > 0 else None
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        **_prefetch_kwargs(num_workers, prefetch_factor),
        pin_memory=pin_memory,
        drop_last=True,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(seed),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        **_prefetch_kwargs(num_workers, prefetch_factor),
        pin_memory=pin_memory,
        drop_last=False,
        persistent_workers=(num_workers > 0),
        worker_init_fn=worker_init_fn,
        generator=_make_loader_generator(None if seed is None else seed + 1),
    )
    return train_loader, val_loader, train_ds, val_ds


In [10]:
# test: end-to-end patch build + dataloader for both multi-channel and single-channel naming
import tempfile
from segmentation_tools_pt.patches import PatchConfig
from segmentation_tools_pt.dataset_build import build_patch_dataset
rng = np.random.default_rng(0)

# --- multi-channel layout (legacy _ch{i} / _mask names) ---
with tempfile.TemporaryDirectory() as d:
    d = Path(d); (d / 'src' / 'images').mkdir(parents=True); (d / 'src' / 'masks').mkdir(parents=True)
    for i in range(4):
        for ch in range(2):
            cv2.imwrite(str(d / 'src' / 'images' / f's{i}_ch{ch}.png'), rng.integers(0, 256, (80, 80), np.uint8))
        m = np.zeros((80, 80), np.uint8); m[20:60, 20:60] = 255
        cv2.imwrite(str(d / 'src' / 'masks' / f's{i}_mask.png'), m)
    cfg = PatchConfig(patch_size=40, stride=40, min_fg_pixels=1, max_empty_fraction=0.0, val_fraction=0.25)
    build_patch_dataset(str(d / 'src'), str(d / 'out'), cfg=cfg, n_channels=2, n_jobs=1)
    tl, vl, tds, vds = build_dataloaders(str(d / 'out'), batch_size=2, num_workers=0, crop_size=32)
    batch = next(iter(tl))
    assert batch['image'].shape == (2, 3, 32, 32)
    assert batch['mask'].shape == (2, 32, 32)
    assert len(tds.get_sample_weights()) == len(tds)

# --- single-channel layout: image/mask patches share the same name ---
with tempfile.TemporaryDirectory() as d:
    d = Path(d); (d / 'src' / 'images').mkdir(parents=True); (d / 'src' / 'masks').mkdir(parents=True)
    for i in range(4):
        cv2.imwrite(str(d / 'src' / 'images' / f's{i}.png'), rng.integers(0, 256, (80, 80), np.uint8))
        m = np.zeros((80, 80), np.uint8); m[20:60, 20:60] = 255
        cv2.imwrite(str(d / 'src' / 'masks' / f's{i}.png'), m)
    cfg = PatchConfig(patch_size=40, stride=40, min_fg_pixels=1, max_empty_fraction=0.0, val_fraction=0.25)
    build_patch_dataset(str(d / 'src'), str(d / 'out'), cfg=cfg, n_channels=1, n_jobs=1)
    tl, vl, tds, vds = build_dataloaders(str(d / 'out'), batch_size=2, num_workers=0, crop_size=32,
                                         n_source_channels=1, channel_names=['ch0'])
    batch = next(iter(tl))
    assert batch['image'].shape == (2, 1, 32, 32)
    assert batch['mask'].shape == (2, 32, 32)
    # filenames in images/ and masks/ should match exactly
    for split in ['train', 'val']:
        assert sorted(p.name for p in (d / 'out' / split / 'images').glob('*.png')) == \
               sorted(p.name for p in (d / 'out' / split / 'masks').glob('*.png'))

In [11]:
#| test
# test: prefetch knob — inert without workers, honored with workers (patch + direct modes).
# Attribute-level contract only: iterating with live workers is environment-dependent
# (sandbox/CI signal limits); torch itself owns the prefetch machinery.
import tempfile

with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    for split, n in (('train', 3), ('val', 2)):
        (d / split / 'images').mkdir(parents=True); (d / split / 'masks').mkdir(parents=True)
        for i in range(n):
            cv2.imwrite(str(d / split / 'images' / f'p{i}.png'), rng.integers(0, 256, (32, 32), np.uint8))
            m = np.zeros((32, 32), np.uint8); m[8:24, 8:24] = 255
            cv2.imwrite(str(d / split / 'masks' / f'p{i}.png'), m)
    # SegmentationDataset reads {split}.txt id lists (normally written by build_patch_dataset)
    (d / 'train.txt').write_text('p0 p1 p2'); (d / 'val.txt').write_text('p0 p1')
    tr, va, _, _ = build_dataloaders(str(d), batch_size=2, num_workers=0, crop_size=32, prefetch_factor=4,
                                     n_source_channels=1, channel_names=['ch0'])
    assert tr.prefetch_factor is None and va.prefetch_factor is None  # knob inert without workers
    tr2, va2, _, _ = build_dataloaders(str(d), batch_size=2, num_workers=2, crop_size=32, prefetch_factor=1,
                                       n_source_channels=1, channel_names=['ch0'])
    assert tr2.prefetch_factor == 1 and va2.prefetch_factor == 1

    with tempfile.TemporaryDirectory() as d2:  # direct mode: flat raw_dir with images/ + masks/
        d2 = Path(d2); (d2 / 'images').mkdir(); (d2 / 'masks').mkdir()
        for i in range(3):
            cv2.imwrite(str(d2 / 'images' / f'p{i}.png'), rng.integers(0, 256, (32, 32), np.uint8))
            m = np.zeros((32, 32), np.uint8); m[8:24, 8:24] = 255
            cv2.imwrite(str(d2 / 'masks' / f'p{i}.png'), m)
        trd, vad, _, _ = build_direct_dataloaders(
            str(d2), ['p0', 'p1'], ['p2'], batch_size=1, num_workers=2, crop_size=32,
            n_source_channels=1, channel_names=['ch0'], mask_suffix='', prefetch_factor=3)
        assert trd.prefetch_factor == 3 and vad.prefetch_factor == 3
        next(iter(trd))

## Multilabel through the real augmentation pipeline

The tests above use `transform=None`, which skips albumentations entirely. The real risk is in `get_train_transforms`/`get_val_transforms` -- crop, affine, and `ToTensorV2` all need to treat a `(H,W,3)` mask the same way they treat a `(H,W,1)` one. Test both `build_dataloaders` (patch-cache mode, `mask_mode='stacked'`) and `build_direct_dataloaders` (raw_dir mode, `mask_mode='per_class_files'`) end to end.

In [12]:
# test: build_dataloaders with mask_mode='stacked' through the real transform pipeline
with tempfile.TemporaryDirectory() as d:
    d = _write_patch_cache(d, n=4, size=40)
    for i in range(4):
        stacked = np.zeros((40, 40, 3), np.uint8)
        stacked[8:32, 8:32, 0] = 255
        stacked[16:24, 16:24, 1] = 255
        stacked[10:30, 10:30, 2] = 255
        cv2.imwrite(str(d / 'train' / 'masks' / f's{i}.png'), stacked)
    tl, vl, tds, vds = build_dataloaders(str(d), batch_size=2, num_workers=0, crop_size=32,
                                          n_source_channels=1, channel_names=['ch0'],
                                          mask_mode='stacked', mask_channels=3)
    batch = next(iter(tl))
    assert batch['image'].shape == (2, 1, 32, 32)
    assert batch['mask'].shape == (2, 3, 32, 32)
    assert batch['mask'].dtype == torch.float32
    assert set(torch.unique(batch['mask']).tolist()) <= {0.0, 1.0}  # still hard {0,1} after crop/affine/normalize

# test: build_direct_dataloaders with mask_mode='per_class_files' through the real transform pipeline
with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d / 'images').mkdir(parents=True); (d / 'masks').mkdir(parents=True)
    chip = np.zeros((40, 40), np.uint8); chip[8:32, 8:32] = 255
    void = np.zeros((40, 40), np.uint8); void[16:24, 16:24] = 255
    lifted = np.zeros((40, 40), np.uint8); lifted[10:30, 10:30] = 255
    for i in range(4):
        cv2.imwrite(str(d / 'images' / f's{i}.png'), rng.integers(0, 256, (40, 40), np.uint8))
        cv2.imwrite(str(d / 'masks' / f's{i}_chip.png'), chip)
        cv2.imwrite(str(d / 'masks' / f's{i}_void.png'), void)
        cv2.imwrite(str(d / 'masks' / f's{i}_lifted.png'), lifted)
    ids = [f's{i}' for i in range(4)]
    tl, vl, tds, vds = build_direct_dataloaders(
        str(d), ids[:3], ids[3:], batch_size=2, num_workers=0, crop_size=32,
        n_source_channels=1, channel_names=['ch0'],
        mask_mode='per_class_files', class_names=['chip', 'void', 'lifted'],
    )
    batch = next(iter(tl))
    assert batch['image'].shape == (2, 1, 32, 32)
    assert batch['mask'].shape == (2, 3, 32, 32)
    assert set(torch.unique(batch['mask']).tolist()) <= {0.0, 1.0}

print('Multilabel pipeline ✓  build_dataloaders + build_direct_dataloaders produce (B,K,H,W) {0,1} masks')

Multilabel pipeline ✓  build_dataloaders + build_direct_dataloaders produce (B,K,H,W) {0,1} masks


In [13]:
#| hide
import nbdev; nbdev.nbdev_export()